# 🔬 KLA Hackathon — NAFNet-SR Image Restoration
**AI-Based Restoration of Degraded Semiconductor Inspection Images**

Run all cells top-to-bottom. Expected training time: ~2h (T4), ~45min (A100).

**FIRST**: Go to `Runtime → Change runtime type → GPU`

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Check GPU
# ─────────────────────────────────────────────
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Mount Google Drive
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Clone repo & install dependencies
# ─────────────────────────────────────────────
import os

# ⚙️ EDIT THIS: your GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/kla-image-restoration.git'
REPO_DIR = '/content/kla_restoration'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -r requirements.txt -q
print('Setup complete!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Set dataset paths
# ─────────────────────────────────────────────
# Option A: Data already extracted on Drive
GT_DIR   = '/content/drive/MyDrive/kla_data/train/GT'
LR_DIR   = '/content/drive/MyDrive/kla_data/train/NoisyLR'
TEST_DIR = '/content/drive/MyDrive/kla_data/test_noisyLR/NoisyLR'

# Option B: Extract zip files from Drive to /content (faster I/O)
# !unzip -q /content/drive/MyDrive/train.zip -d /content/kla_data/
# !unzip -q /content/drive/MyDrive/test_noisyLR.zip -d /content/kla_data/
# GT_DIR   = '/content/kla_data/train/train/GT'
# LR_DIR   = '/content/kla_data/train/train/NoisyLR'
# TEST_DIR = '/content/kla_data/test_noisyLR/NoisyLR'

import os
gt_count = len(list(__import__('pathlib').Path(GT_DIR).glob('*.npy')))
lr_count = len(list(__import__('pathlib').Path(LR_DIR).glob('*.npy')))
print(f'GT files:    {gt_count}')
print(f'NoisyLR:     {lr_count}')
print('Paths OK!' if gt_count > 0 and lr_count > 0 else 'ERROR: No files found — check paths!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Quick data inspection
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import os, glob

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    idx = i * 200  # sample every 200 files
    gt = np.load(gt_files[min(idx, len(gt_files)-1)])
    lr = np.load(lr_files[min(idx, len(lr_files)-1)])
    lr_vis = np.clip(lr, 0, 1)  # clip for display only

    axes[0][i].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][i].set_title(f'GT #{min(idx, len(gt_files)-1)} (256×256)', fontsize=9)
    axes[0][i].axis('off')

    axes[1][i].imshow(lr_vis, cmap='gray', vmin=0, vmax=1)
    axes[1][i].set_title(f'NoisyLR #{min(idx, len(lr_files)-1)} (128×128)\nmax={lr.max():.2f}', fontsize=9)
    axes[1][i].axis('off')

plt.suptitle('Sample Training Pairs (top: GT, bottom: NoisyLR)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/sample_pairs.png')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Train!
# ─────────────────────────────────────────────
# ⚙️ Adjust batch_size: 16 for A100, 8 for T4
BATCH_SIZE = 8   # T4 safe
EPOCHS = 200

!python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers 2 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir ./weights \
    --log_dir ./logs

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Launch TensorBoard
# ─────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir ./logs

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Run evaluation on validation set (with metrics)
# ─────────────────────────────────────────────
import os, sys
sys.path.insert(0, '.')

# Build a mini val split directory for evaluate.py
# (evaluate.py works with flat directory of NoisyLR .npy files)
os.makedirs('/content/val_outputs', exist_ok=True)

# Run evaluation on validation NoisyLR images with GT for metrics
!python evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /content/val_outputs \
    --gt_dir {GT_DIR} \
    --weights ./weights/best_model.pt \
    --batch_size 8

import json
with open('/content/val_outputs/metrics.json') as f:
    m = json.load(f)
print('\n=== FINAL METRICS ===')
for k, v in m.items():
    print(f'  {k}: {v}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Run inference on official test set
# ─────────────────────────────────────────────
import os
os.makedirs('/content/test_outputs', exist_ok=True)

!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs \
    --weights ./weights/best_model.pt \
    --batch_size 8

import json
with open('/content/test_outputs/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Visualise before/after results
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob, os, random

gt_files   = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files   = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))
out_files  = sorted(glob.glob('/content/val_outputs/*.npy'))

# Pick 4 interesting samples (choose ones with heavy speckle)
lr_maxvals = [(np.load(f).max(), f) for f in lr_files[:200]]
lr_maxvals.sort(reverse=True)
sample_stems = [os.path.splitext(os.path.basename(f))[1-1] for _, f in lr_maxvals[:4]]
# Fallback to random
sample_files = [f for _, f in lr_maxvals[:4]]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
titles = ['NoisyLR Input (128×128)', 'NAFNet-SR Output (256×256)', 'Ground Truth (256×256)']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for row, lr_path in enumerate(sample_files):
    stem = os.path.splitext(os.path.basename(lr_path))[0]
    gt_path  = os.path.join(GT_DIR, f'{stem}.npy')
    out_path = f'/content/val_outputs/{stem}.npy'

    lr_arr  = np.load(lr_path)
    gt_arr  = np.load(gt_path) if os.path.exists(gt_path) else np.zeros((256,256))
    out_arr = np.load(out_path) if os.path.exists(out_path) else np.zeros((256,256))

    for col, (arr, title, color) in enumerate(zip([lr_arr, out_arr, gt_arr], titles, colors)):
        axes[row][col].imshow(np.clip(arr, 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row][col].set_title(f'{title}\n[{arr.min():.2f}, {arr.max():.2f}]', 
                                  fontsize=10, color=color, fontweight='bold')
        axes[row][col].axis('off')

    axes[row][0].set_ylabel(f'Sample {stem}', fontsize=10, rotation=90, labelpad=15)

plt.suptitle('NAFNet-SR: Before → After → Ground Truth', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/before_after_comparison.png')
print('Use this image in your PPT Slide 6!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Save outputs & weights to Drive
# ─────────────────────────────────────────────
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/kla_submission'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy model weights
shutil.copy('./weights/best_model.pt', f'{DRIVE_OUTPUT}/best_model.pt')
print('Saved weights to Drive')

# Copy test outputs
shutil.copytree('/content/test_outputs', f'{DRIVE_OUTPUT}/test_outputs', dirs_exist_ok=True)
print('Saved test outputs to Drive')

# Copy comparison image
shutil.copy('/content/before_after_comparison.png', f'{DRIVE_OUTPUT}/before_after_comparison.png')
print('Saved comparison image to Drive')

print(f'\nAll saved to: {DRIVE_OUTPUT}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Run with TTA for best quality (optional)
# ─────────────────────────────────────────────
os.makedirs('/content/test_outputs_tta', exist_ok=True)

!python evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs_tta \
    --weights ./weights/best_model.pt \
    --tta \
    --batch_size 1

import json
with open('/content/test_outputs_tta/metrics.json') as f:
    print('TTA results:')
    print(json.dumps(json.load(f), indent=2))